# Bài toán 3 — Thảo luận, demo và báo cáo

## 9. Giải thích và thảo luận

Mô hình được chọn là **mùa vụ mức gốc cố định có hiệu ứng thứ trong tuần**. Dự báo cho mỗi ngày bằng **mức trung bình ngày của năm đầy đủ gần nhất × hệ số mùa vụ tháng–ngày × hệ số theo thứ trong tuần**. Các hệ số được ước lượng riêng cho Revenue và COGS từ tập train. Cùng một cặp tháng–ngày có thể cho dự báo khác nhau nếu rơi vào các thứ khác nhau.

Trên tập test, ngày **02/03/2022** có sai số tuyệt đối lớn nhất ở cả hai chỉ tiêu. Dự báo thấp hơn thực tế khoảng **4,013 triệu** với Revenue và **3,481 triệu** với COGS. Kết quả cho thấy mô hình có thể bỏ lỡ những biến động mạnh trong ngày. Ba cột của `sales.csv` cho phép xác định thời điểm và mức sai lệch, còn nguyên nhân nghiệp vụ cần dữ liệu bổ sung.

Trên ba lượt validation, phiên bản có hiệu ứng thứ giảm MAE cho cả hai chỉ tiêu so với mô hình mùa vụ gốc ở từng lượt. So với baseline cuộc thi, phiên bản này có MAE thấp hơn ở hai lượt đầu. Ở lượt thứ ba, baseline đạt MAE thấp hơn cho cả hai chỉ tiêu.

## 10. Demo dự báo trong notebook

Đổi hai mốc `demo_start` và `demo_end` để xem dự báo Revenue và COGS trong một khoảng của tập test. Bảng đọc dự báo được chọn từ `forecast_predictions.csv` do notebook 02 tạo. Bảng mặc định hiển thị bảy ngày đầu của tập test.

Trong ứng dụng Streamlit của nhóm, tab **Dự báo** dùng tệp `forecast_2021-07-01.csv` xuất từ notebook 03 để hiển thị và tải kết quả.

In [1]:
from pathlib import Path
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

sales_path = Path("data/raw/sales.csv")
start = Path(os.environ.get("RETAIL_ANALYTICS_ROOT", Path.cwd())).resolve()
project_root = next(
    (folder for folder in (start, *start.parents) if (folder / sales_path).is_file()),
    None,
)
if project_root is None:
    raise FileNotFoundError(
        "Không tìm thấy data/raw/sales.csv. Hãy đặt notebook trong dự án "
        "hoặc trỏ RETAIL_ANALYTICS_ROOT về thư mục gốc."
    )

prediction_path = project_root / "data/model/du_bao/forecast_predictions.csv"
if not prediction_path.is_file():
    raise FileNotFoundError("Chạy 02_mo_hinh.ipynb để tạo forecast_predictions.csv.")

predictions = pd.read_csv(
    prediction_path, parse_dates=["Date", "TrainEnd"], float_precision="round_trip",
)
targets = ["Revenue", "COGS"]

contest_name = "Baseline cuộc thi"
fixed_name = "Mùa vụ, mức gốc cố định"
selected_label = "Phương pháp được chọn"


def prediction_frame(rows):
    return rows.set_index("Date")[targets].sort_index()


def test_forecast_for(method):
    return prediction_frame(predictions.loc[
        predictions["Split"].eq("test") & predictions["Method"].eq(method)
    ])

selected_test_forecast = test_forecast_for(selected_label)

demo_start = "2021-07-02"
demo_end = "2021-07-08"
display(selected_test_forecast.loc[demo_start:demo_end].rename(
    columns={"Revenue": "Revenue dự báo", "COGS": "COGS dự báo"},
))

,Revenue dự báo,COGS dự báo
Date,,
2021-07-02,4209737.42,4026908.63
2021-07-03,3725039.19,3573674.29
2021-07-04,1240758.54,1091703.73
2021-07-05,1610694.44,1485779.33
2021-07-06,1787767.69,1673779.67
2021-07-07,2186756.77,2068807.26
2021-07-08,1977447.94,1889918.16


## 11. Nội dung đưa vào báo cáo

Nhóm sử dụng 3.833 quan sát theo ngày để khảo sát biến động, xây dựng và so sánh các phương pháp dự báo Revenue và COGS. Mô hình mùa vụ mức gốc cố định có hiệu ứng thứ trong tuần được chọn theo MAE trung bình trên ba lượt validation, mỗi lượt dự báo 548 ngày. Mô hình đạt MAE validation lần lượt là **1,182** và **1,018** triệu đơn vị tiền tệ mỗi ngày cho Revenue và COGS.

Sau khi chọn mô hình, nhóm ước lượng lại các hệ số bằng dữ liệu đến hết **01/07/2021** và đánh giá trên **548 ngày từ 02/07/2021 đến 31/12/2022**.

| Chỉ tiêu | MAE | RMSE | R² | MAE giảm so với baseline cuộc thi |
|---|---:|---:|---:|---:|
| Revenue | 0,577 | 0,798 | 0,736 | 16,4% |
| COGS | 0,527 | 0,734 | 0,712 | 22,4% |

MAE và RMSE tính theo triệu đơn vị tiền tệ mỗi ngày. Mức giảm MAE được tính so với baseline cuộc thi trên cùng kỳ test. Bảng đánh giá ở mục 7 và biểu đồ, sai số theo ngày ở mục 8 là minh chứng cho báo cáo.

## 12. Thuyết trình và vấn đáp

**Mạch trình bày.** Nêu hai chỉ tiêu và mốc dự báo, giải thích cách chia validation và test, đọc bảng so sánh, rồi mở biểu đồ và chọn một ngày có sai số lớn để minh họa hạn chế.

| Câu hỏi có thể gặp | Cách giải thích |
|---|---|
| Vì sao chia theo thời gian? | Tại mỗi mốc dự báo, nhóm chỉ dùng lịch sử có đến mốc đó để tính hệ số. Tách theo thời gian tái hiện điều kiện này. |
| Vì sao chọn mô hình có hiệu ứng thứ trong tuần? | Mô hình có MAE trung bình thấp nhất cho cả Revenue và COGS trên ba lượt validation. Thứ trong tuần bổ sung thông tin cho mẫu mùa vụ theo cặp tháng–ngày. |
| Vì sao có cả MAE, RMSE và R²? | MAE cho biết độ lớn sai số trung bình. RMSE nhấn mạnh sai số lớn. R² so sánh mô hình với mức tham chiếu là trung bình thực tế của giai đoạn chấm. |
| Mô hình có hạn chế gì? | Mô hình dùng mức gốc cố định trong suốt kỳ dự báo, nên có thể sai nhiều khi mức doanh thu hoặc giá vốn thay đổi rõ rệt hay có ngày tăng đột biến. |